In [78]:
import os
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import re
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, words, wordnet
from gensim.utils import tokenize

In [ ]:
## Load path from .env file
## path should point to a folder that contains all .txt files in "women", "mixed", and "men" subfolders
load_dotenv()
path = os.getenv('dir')

In [ ]:
## Read all transcripts
def read_text_files(path=path):

    files = []
    transcripts = []
    genders = []

    for gender in ['women', 'mixed', 'men']:
        for file in os.listdir(os.path.join(path, gender)):
            if 'txt' in file:
                try:
                    with open(os.path.join(path, gender, file), 'r', encoding='utf-8', errors='ignore') as f:
                        transcript = f.read()
                        files.append(file)
                        transcripts.append(transcript)
                        genders.append(gender)
                except:
                    print('error for ', file)

    df = pd.DataFrame({'id' : files, 'transcript' : transcripts, 'gender' : genders})

    return df

df = read_text_files()

In [ ]:
## Words to count as in-vocab
stop = stopwords.words('english')
stop.extend(['dr', 'mrs', 'mr', 'ms', 'uh', 'um', 'umm'])
stop.extend(['be', 'is', 'are', 'do', 'have', 'had', 'having', 'can', 'could', 'may', 'might', 'shall', 
             'should', 'will', 'would', 'whether', 'without', 'anything', 'something', 'nothing', 'else'])
legal = ['voir', 'per']
contractions = ['aight', 'aint', 'arent', 'cant', 'cmon', 'couldve', 'couldnt', 'couldntve', 'darent', 'didnt', 'doesnt', 'dont', 'everybodys', 'everyones', 'everythings', 'hadnt', 'hadve', 'hasnt', 'havent', 'hed', 'hell', 'hes', 'heres', 'howd', 'howll', 'howre', 'hows', 'id', 'idve', 'ill', 'im', 'ive', 'isnt', 'itd', 'itll', 'its', 'lets', 'maam', 'mightnt', 'mightve', 'nothings', 'oclock', 'oughtve', 'oughtnt', 'shant', 'shed', 'shell', 'shes', 'shouldve', 'shouldnt', 'shouldntve', 'somebodys', 'someones', 'somethings', 'sore', 'sos', 'sove', 'thatll', 'thatre', 'thats', 'thatd', 'thered', 'therell', 'therere', 'theres', 'thesere', 'theseve', 'theyd', 'theydve', 'theyll', 'theyre', 'theyve', 'til', 'wasnt', 'wed', 'wedve', 'well', 'were', 'weve', 'werent', 'whatd', 'whatll', 'whatre', 'whats', 'whatve', 'whens', 'whered', 'wherell', 'wheres', 'whod', 'whodve', 'wholl', 'whore', 'whos', 'whove', 'whyd', 'whyre', 'whys', 'wont', 'wouldve', 'wouldnt', 'wouldntve', 'yall', 'yesm', 'yknow', 'youd', 'youll', 'youre', 'youve', 'others', 'anytime']

trial = ([
    ## angelina rodriguez
    'angelina', 'rodriguez', 
    'jose', 'francisco', 'frank', 
    'san', 'luis', 'obispo', 'california', 'los', 'angeles', 'la', 'montebello', 'marconi', 'santa', 'barbara', 'paso',
    'sortino', 'joe', 'holmes', 'brian', 'steinwand', 'houchin', 'mejia',
    'mickey', 'marracino', 'matt', 'morones', 'palmira', 'gorham', 'palmiras', 'gorhams', 'conrad', 'apodaca', 'aguilar', 
    'gwendolyn', 'stephen', 'sharpe', 'rebecca', 'perkins', 'charles', 'chad', 'holloway', 'shirley', 'knauss',
    'coers', 'wanda', 'moats', 'puschner', 'richard', 'clark',
    'oleander', 'gerber',

    ## antoinette frank
    'antoinette', 'antoinettes', 'frank', 'roger', 'lacaze',
    'chau', 'quoc', 'cuong', 'vu', 'ronnie', 'williams', 
    'kim', 'anh', 'orleans', 'bullard',
    'larre', 'jenkins',
    'farve', 'demma', 'rantz', 'sgt', 'mcgary', 'dermott', 'dugger'

    ## belinda magana
    'belinda', 'magana', 'naresh', 'narine', 'narines', 'maganas', 'belindas'
    'malachi', 'armando', 'salcido'
    'bernardino', 
    'calhoun', 'markson', 'catlett', 'belter', 'ritt', 
    'exum', 'bryant', 'trenkle', 'lofton', 'alejandre', 'camacho',

    ## blanche taylor moore
    'blanche', 'taylor', 'moore',
    'mcentire', 'loflin', 'rabil', 
    'kroger', 'alamance',
    'garvin', 'thomas', 'otis', 'mcneil', 'parker', 'davis', 'kiser', 'vincent', 'guinn', 'doris', 'pender', 
]) ## proper nouns related to transcript

words_big = set([word.lower() for word in words.words()])

In [ ]:
## Count malwords: words where alpha and non-alpha characters are together 
## (and the word is not a dictionary word like "1st")
df['transcript_cleaned'] = df['transcript'].str.replace('-', ' ', regex=False).str.replace('/', ' ', regex=False) ## replace hyphen, backslash with space
df['malwords'] = df['transcript_cleaned'].str.replace("'|’", " ", regex=True) ## replace apostrophe with space
df['malwords'] = df['malwords'].str.replace(r'(?<!\w)([A-Z])\.', r'\1', regex=True) ## remove periods from acronyms
df['malwords'] = df['malwords'].str.replace('a.m.', 'am', regex=False).str.replace('p.m.', 'pm', regex=False) ## manualy replace am/pm
df['malwords'] = df['malwords'].str.split() ## split on whitespace
df['malwords'] = df['malwords'].apply(lambda x: [word.strip(string.punctuation) for word in x]) ## strip punctuation from either end of character
df['malwords'] = df['malwords'].apply(lambda x: [word for word in x if not all(char.isalpha() for char in word) and any(char.isalpha() for char in word)]) ## count words with mixed alpha/non-alpha characters
df['malwords'] = df['malwords'].apply(lambda x: [word for word in x if word.lower() not in wordnet.words()]) ## filter out dictionary words

## Count oov (out-of-vocab) words: words that do not exist in dictionaries or trial context
df['transcript_tokenized'] = df['transcript_cleaned'].apply(lambda x: list(tokenize(x))) ## split on whitespace and punctuation
df['transcript_oov'] = df['transcript_tokenized'].apply(lambda x: [word for word in x if word.lower() not in stop]) ## remove stopwords
df['transcript_oov'] = df['transcript_oov'].apply(lambda x: [word for word in x if word.lower() not in legal]) ## remove contractions
df['transcript_oov'] = df['transcript_oov'].apply(lambda x: [word for word in x if word.lower() not in contractions]) ## contractions
df['transcript_oov'] = df['transcript_oov'].apply(lambda x: [word for word in x if len(wordnet.synsets(word.lower())) == 0]) ## non-dictionary words (small dictionary)
df['transcript_oov'] = df['transcript_oov'].apply(lambda x: [word for word in x if word.lower() not in words_big]) ## non-dictionary words (big dictionary)
df['transcript_oov_nontrial'] = df['transcript_oov'].apply(lambda x: [word for word in x if word.lower() not in trial]) ## trial-specific terms (proper nouns)

## Summary statistics
df['length_total'] = df['transcript_tokenized'].apply(len)
df['length_oov'] = df['transcript_oov'].apply(len)
df['length_oov_nontrial'] = df['transcript_oov_nontrial'].apply(len)
df['length_malwords'] = df['malwords'].apply(len)

df['malword_ratio'] = df['length_malwords'] / df['length_total']
df['oov_ratio'] = df['length_oov'] / df['length_total']
df['oov_nontrial_ratio'] = df['length_oov_nontrial'] / df['length_total']

In [ ]:
## Get examples of OOV words and malwords
df['oov_list_common'] = df['transcript_oov'].apply(nltk.FreqDist).apply(lambda x: x.most_common(100))
df['oov_list_rare'] = df['transcript_oov'].apply(nltk.FreqDist).apply(lambda x: x.most_common(10000)[-100:])
df['malwords_list'] = df['malwords'].apply(nltk.FreqDist).apply(lambda x: x.most_common(100))

In [ ]:
## Export
df.sort_values('oov_ratio', ascending=False)[['id', 'gender', 'oov_list_common', 'oov_list_rare', 'malwords_list', 'length_total', 'length_oov', 'length_malwords', 'oov_ratio', 'malword_ratio']].to_csv('data_quality_check.csv', index=False)